Importar bibliotecas e carregar os dados

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# 1. Carregar o dataset original (UCI) a partir do .txt
#    - separador ';'
#    - valores em falta marcados com '?'
colunas_num = ['Global_active_power', 'Global_reactive_power', 'Voltage', 'Global_intensity',
               'Sub_metering_1', 'Sub_metering_2', 'Sub_metering_3']
df = pd.read_csv('household_power_consumption.txt', sep=';', na_values='?',
                 dtype={c: 'float64' for c in colunas_num})

# 2. Criar a coluna Datetime (formato europeu dd/mm/aaaa) e defini-la como index
df['Datetime'] = pd.to_datetime(df['Date'] + ' ' + df['Time'], format='%d/%m/%Y %H:%M:%S')
df = df.drop(columns=['Date', 'Time']).set_index('Datetime')
# Global_active_power: potência ativa total consumida pela casa (kW), média de cada minuto

print(f"{len(df):,} registos de {df.index.min()} a {df.index.max()}")
print(f"Minutos em falta: {df['Global_active_power'].isna().sum():,} "
      f"({df['Global_active_power'].isna().mean():.2%})")

2,075,259 registos de 2006-12-16 17:24:00 a 2010-11-26 21:02:00
Minutos em falta: 25,979 (1.25%)


In [2]:
# 3. Agregar os dados à hora (kW médio na hora = kWh consumidos nessa hora)
# Limpa o ruído e prepara os dados para cruzar com a API Solar
gap = df['Global_active_power']
df_hourly = gap.resample('h').mean()

# Horas com menos de 30 minutos válidos são consideradas em falta
minutos_validos = gap.resample('h').count()
df_hourly[minutos_validos < 30] = np.nan

# Preencher apenas falhas curtas (até 3h) por interpolação; falhas longas ficam em falta
df_hourly = df_hourly.interpolate(limit=3, limit_area='inside')

# Matriz dias x 24 horas, apenas com dias completos (os perfis são construídos dia a dia)
matriz = (df_hourly.to_frame('kW')
          .assign(dia=lambda x: x.index.normalize(), hora=lambda x: x.index.hour)
          .pivot(index='dia', columns='hora', values='kW')
          .dropna())

print(f"Dias completos usados: {len(matriz)} de {df_hourly.index.normalize().nunique()}")

Dias completos usados: 1419 de 1442


In [ ]:
matriz

Metodologia dos perfis

Em vez de multiplicar o consumo original por constantes, cada perfil é construído a partir dos dados reais:

1. **Carga base** (frigorífico, standby, router...): mínimo horário de cada dia. Existe sempre, mesmo sem ninguém em casa.
2. **Carga ativa**: o resto do consumo, que depende do comportamento das pessoas.
3. Cada perfil define uma **rotina horária** (pesos relativos de atividade em cada hora). A energia ativa diária real é **redistribuída** pelas horas segundo essa rotina, mantendo a **variabilidade real** de cada hora (picos, dias atípicos, sazonalidade, férias).
4. Ajustes específicos por perfil: mais/menos energia (nº de pessoas, tempo em casa), equipamentos extra, ruído e limite de potência contratada.

| Perfil | Rotina (dias úteis) | Ajustes |
|---|---|---|
| 1 - Original | Agregado real (UCI) | - |
| 2 - Escritório | Fora 9h-18h, picos 6h-8h e 19h-22h | -15% energia nos dias úteis; fim de semana = original |
| 3 - Família numerosa | Pico manhã (escola), regresso 16h, jantar | +70% energia ativa, +40% base, ruído com memória, 9.2 kVA |
| 4 - Trabalhador noturno | Turno 22h-7h (seg-sex), dorme 9h-15h | -5% energia, folgas com rotina deslocada para a noite |
| 5 - Home office | Em casa todo o dia, almoço em casa | +15% energia, PC/monitores 9h-18h, aquecimento/arrefecimento sazonal |

In [ ]:
# 4. Decompor o consumo em carga base + carga ativa
horas = np.arange(24)
dias = matriz.index
D = len(dias)
fim_de_semana = np.asarray(dias.dayofweek >= 5)

base = matriz.min(axis=1).to_numpy()            # kW sempre ligados (D,)
ativa = matriz.to_numpy() - base[:, None]       # kW dependentes de comportamento (D, 24)
energia_ativa = ativa.sum(axis=1)               # kWh ativos por dia

# Forma média do agregado original (dias úteis vs fim de semana)
forma_orig_semana = ativa[~fim_de_semana].mean(axis=0)
forma_orig_fds = ativa[fim_de_semana].mean(axis=0)
forma_orig = np.where(fim_de_semana[:, None], forma_orig_fds, forma_orig_semana)

# Variabilidade real: quanto cada hora se afasta da média dessa hora (1 = normal)
# Limitada para evitar picos irrealistas nas horas de consumo médio muito baixo
variabilidade = np.clip(ativa / forma_orig, 0.25, 4.0)

# Dias em que a casa original esteve praticamente vazia (férias, ausências)
em_casa = energia_ativa > 0.25 * np.median(energia_ativa)

POTENCIA_CONTRATADA = 6.9  # kVA, potência típica de uma habitação


def rotina(blocos, padrao=0.1):
    # Vetor de 24 pesos. blocos = {(hora_inicio, hora_fim_exclusiva): peso}. 
    w = np.full(24, padrao, dtype=float)
    for (inicio, fim), valor in blocos.items():
        w[inicio:fim] = valor
    return w


def por_dia(semana, fds):
    # Matriz (D, 24): uma rotina para dias úteis e outra para o fim de semana. 
    return np.where(fim_de_semana[:, None], fds, semana)


def gerar_perfil(pesos, fator_energia=1.0, fator_base=1.0, extra=None, ruido=None,
                 limite=POTENCIA_CONTRATADA):
    """
    pesos: matriz (D, 24) com a rotina horária do perfil (valores relativos)
    fator_energia: escalar ou (D,), multiplica a energia ativa diária
    fator_base: multiplica a carga base (ex.: arca congeladora extra)
    extra: matriz (D, 24) em kW somada no fim (ex.: computador do teletrabalho)
    ruido: matriz (D, 24) multiplicativa
    """
    padrao = pesos * variabilidade
    padrao = padrao / padrao.sum(axis=1, keepdims=True)
    energia = energia_ativa * np.broadcast_to(fator_energia, (D,))
    consumo = base[:, None] * fator_base + padrao * energia[:, None]
    if extra is not None:
        consumo = consumo + extra
    if ruido is not None:
        consumo = consumo * ruido
    return np.clip(consumo, 0.05, limite)

5 Perfis de Utilizador

In [ ]:
perfis = {}

# Perfil 1: O Consumo Base (Original)
perfis['Perfil_1_Original'] = matriz.to_numpy()

# Perfil 2: Trabalhador de Escritório
# Dias úteis: acorda às 6h, sai às 8h-9h, regressa às 18h; só a carga base fica ligada durante o dia
# Fim de semana: rotina igual ao agregado original
escritorio_semana = rotina({(0, 6): 0.12, (6, 8): 1.3, (8, 9): 0.5, (9, 18): 0.05,
                            (18, 19): 0.9, (19, 22): 1.6, (22, 23): 1.0, (23, 24): 0.5})
perfis['Perfil_2_Escritorio'] = gerar_perfil(
    por_dia(escritorio_semana, forma_orig_fds),
    fator_energia=np.where(fim_de_semana, 1.0, 0.85),   # menos horas em casa nos dias úteis
)

# Perfil 3: Família Numerosa (2 adultos + 3 crianças)
# Mais pessoas -> mais energia ativa (máquinas, banhos, cozinha) e mais carga base (arca extra)
# Ruído multiplicativo: componente diária com memória (dias seguidos parecidos) + componente horária
rng = np.random.default_rng(42)
z = np.zeros(D)
eps = rng.normal(0, 0.08, D)
for i in range(1, D):
    z[i] = 0.7 * z[i - 1] + eps[i]
ruido_familia = np.exp(z[:, None] + rng.normal(0, 0.12, (D, 24)))

familia_semana = rotina({(0, 6): 0.15, (6, 7): 1.2, (7, 8): 1.8, (8, 9): 0.8, (9, 12): 0.35,
                         (12, 14): 0.6, (14, 16): 0.35, (16, 19): 1.0, (19, 22): 1.8,
                         (22, 23): 1.0, (23, 24): 0.4})
familia_fds = rotina({(0, 7): 0.15, (7, 8): 0.6, (8, 11): 1.2, (11, 14): 1.5, (14, 19): 1.0,
                      (19, 22): 1.6, (22, 23): 1.0, (23, 24): 0.5})
perfis['Perfil_3_Familia'] = gerar_perfil(
    por_dia(familia_semana, familia_fds),
    fator_energia=1.7, fator_base=1.4, ruido=ruido_familia, limite=9.2,
)

# Perfil 4: Trabalhador Noturno
# Turno 22h-7h com entrada de segunda a sexta à noite (sai de terça a sábado de manhã)
# Depois do turno: refeição/banho, dorme 9h-15h, ativo à tarde/noite, sai às 21h30
# Folgas (dom/seg de dia): mantém o ritmo deslocado -> deita-se tarde e acorda tarde
dow = np.asarray(dias.dayofweek)
trabalha_hoje = dow < 5                        # entra às 22h
saiu_do_turno = (dow >= 1) & (dow <= 5)        # chegou a casa às 7h

madrugada_folga = rotina({(0, 2): 1.0, (2, 3): 0.5}, padrao=0.12)
dia_pos_turno = rotina({(7, 9): 1.2, (9, 15): 0.08, (15, 17): 0.8, (17, 21): 1.4, (21, 22): 1.0})
dia_folga = rotina({(7, 10): 0.12, (10, 11): 0.6, (11, 14): 1.2, (14, 19): 0.9, (19, 22): 1.5})
noite_em_casa = rotina({(22, 23): 1.3, (23, 24): 1.1})

noturno = np.empty((D, 24))
noturno[:, 0:7] = np.where(saiu_do_turno[:, None], 0.02, madrugada_folga[0:7])
noturno[:, 7:22] = np.where(saiu_do_turno[:, None], dia_pos_turno[7:22], dia_folga[7:22])
noturno[:, 22:24] = np.where(trabalha_hoje[:, None], 0.02, noite_em_casa[22:24])
perfis['Perfil_4_Noturno'] = gerar_perfil(noturno, fator_energia=0.95)

# Perfil 5: Home Office (teletrabalho nos dias úteis)
# Em casa o dia todo: consumo diurno moderado, almoço cozinhado em casa
# Extra: PC + monitores + router (~120 W) 9h-18h e climatização do escritório consoante o mês
home_semana = rotina({(0, 6): 0.12, (6, 7): 0.5, (7, 9): 1.1, (9, 12): 0.6, (12, 14): 1.3,
                      (14, 18): 0.6, (18, 19): 0.9, (19, 22): 1.4, (22, 23): 0.9, (23, 24): 0.4})
climatizacao = {1: 0.40, 2: 0.32, 3: 0.20, 4: 0.08, 5: 0.0, 6: 0.05,
                7: 0.15, 8: 0.15, 9: 0.0, 10: 0.08, 11: 0.24, 12: 0.40}   # kW médios
horario_trabalho = rotina({(9, 18): 1.0}, padrao=0.0)
dia_teletrabalho = (~fim_de_semana) & em_casa
kw_extra = 0.12 + np.asarray(dias.month.map(climatizacao), dtype=float)
extra_home = (dia_teletrabalho * kw_extra)[:, None] * horario_trabalho

perfis['Perfil_5_HomeOffice'] = gerar_perfil(
    por_dia(home_semana, forma_orig_fds),
    fator_energia=np.where(fim_de_semana, 1.0, 1.15),
    extra=extra_home,
)


# Converter matrizes (D, 24) em séries horárias
def para_serie(m):
    s = pd.DataFrame(m, index=dias, columns=horas).stack()
    s.index = s.index.get_level_values(0) + pd.to_timedelta(s.index.get_level_values(1), unit='h')
    return s


df_simulacao = pd.DataFrame({nome: para_serie(m) for nome, m in perfis.items()}).round(3)
df_simulacao.index.name = 'Datetime'
df_simulacao.head()

Validar os perfis

In [ ]:
# Resumo: consumo diário e distribuição pelas horas do dia (dias úteis)
uteis = df_simulacao[df_simulacao.index.dayofweek < 5]
h = uteis.index.hour
resumo = pd.DataFrame({
    'kWh/dia (média)': df_simulacao.resample('D').sum().replace(0, np.nan).mean(),
    'Pico horário (kW)': df_simulacao.max(),
    '% energia 9h-18h (úteis)': uteis[(h >= 9) & (h < 18)].sum() / uteis.sum() * 100,
    '% energia 22h-7h (úteis)': uteis[(h >= 22) | (h < 7)].sum() / uteis.sum() * 100,
}).round(1)
resumo

Visualizar e Exportar

In [ ]:
nomes = {
    'Perfil_1_Original': 'Perfil 1 (Base)',
    'Perfil_2_Escritorio': 'Perfil 2 (Escritório)',
    'Perfil_3_Familia': 'Perfil 3 (Família numerosa)',
    'Perfil_4_Noturno': 'Perfil 4 (Trabalhador noturno)',
    'Perfil_5_HomeOffice': 'Perfil 5 (Home office)',
}

# 1. Perfil médio por hora do dia (dias úteis vs fim de semana)
fig, axes = plt.subplots(1, 2, figsize=(15, 5), sharey=True)
for ax, (titulo, mask) in zip(axes, [('Dias úteis', df_simulacao.index.dayofweek < 5),
                                     ('Fim de semana', df_simulacao.index.dayofweek >= 5)]):
    media = df_simulacao[mask].groupby(df_simulacao[mask].index.hour).mean()
    for col in media.columns:
        ax.plot(media.index, media[col], marker='o', markersize=3, label=nomes[col])
    ax.set_title(f'Perfil médio horário - {titulo}')
    ax.set_xlabel('Hora do dia')
    ax.set_xticks(range(0, 24, 2))
    ax.grid(True)
axes[0].set_ylabel('Consumo médio (kW)')
axes[0].legend()
plt.tight_layout()
plt.show()

# 2. Uma semana típica (segunda a domingo) para ver a diferença entre perfis
semana_teste = df_simulacao.loc['2008-01-14':'2008-01-20']
plt.figure(figsize=(15, 6))
for col in semana_teste.columns:
    plt.plot(semana_teste.index, semana_teste[col], label=nomes[col], alpha=0.8)
plt.title('Simulação de Consumo de Energia - 5 Perfis Diferentes (14 a 20 jan 2008)')
plt.ylabel('Consumo (kW)')
plt.xlabel('Data / Hora')
plt.legend()
plt.grid(True)
plt.show()

# 3. Exportar o resultado final para CSV
df_simulacao.to_csv('dataset_consumo_simulado.csv')
print(f"Dataset gerado com sucesso! {df_simulacao.shape[0]:,} horas x {df_simulacao.shape[1]} perfis")